<a href="https://colab.research.google.com/github/ReginaStrem/Single-atom-sensors/blob/main/Untitled16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests, csv, time, os
base = "https://nomad-lab.eu/prod/v1/api/v1"
DOIS = ["10.17172/NOMAD/2024.05.10-1"]   # Rivera-Arrieta 2025, CO2 на SAA
s = requests.Session()

def get(method, url, **kw):
    for attempt in range(10):
        try:
            r = s.request(method, url, timeout=600, **kw)
            if r.status_code == 200:
                return r.json()
            print("HTTP", r.status_code, r.text[:150])
        except Exception as ex:
            print("сбой:", type(ex).__name__)
        time.sleep(15 * (attempt + 1))
    return None

for doi in DOIS:
    meta = get("GET", base + "/datasets", params={"doi": doi})
    if not meta or not meta.get("data"):
        print("не нашёл", doi); continue
    ds = meta["data"][0]["dataset_id"]
    tag = doi.split("/")[-1]
    out, state = f"nomad_{tag}.csv", f"cursor_{tag}.txt"
    if os.path.exists(out) and not os.path.exists(state):
        print("уже скачан:", out); continue
    if not os.path.exists(out):
        with open(out, "w", newline="") as f:
            csv.writer(f).writerow(("entry_id", "mainfile", "formula", "E_total_eV"))
    after = open(state).read().strip() if os.path.exists(state) else None
    body = {"owner": "public", "query": {"datasets.dataset_id": ds},
            "pagination": {"page_size": 20},
            "required": {"metadata": "*", "run": {"calculation[-1]": {"energy": "*"}}}}
    n = sum(1 for _ in open(out)) - 1
    print("набор", doi, "→", out)
    while True:
        if after: body["pagination"]["page_after_value"] = after
        d = get("POST", base + "/entries/archive/query", json=body)
        if d is None:
            print("сервер не отвечает — перезапустите ячейку позже, продолжит с места"); break
        rows = []
        for x in d.get("data", []):
            a = x.get("archive", {}); m = a.get("metadata", {})
            try: e = a["run"][0]["calculation"][-1]["energy"]["total"]["value"] / 1.602176634e-19
            except Exception: e = ""
            rows.append((x.get("entry_id", ""), m.get("mainfile", ""),
                         (m.get("optimade") or {}).get("chemical_formula_reduced", ""), e))
        with open(out, "a", newline="") as f: csv.writer(f).writerows(rows)
        n += len(rows); print("  строк:", n)
        after = d.get("pagination", {}).get("next_page_after_value")
        if not after:
            if os.path.exists(state): os.remove(state)
            print("  ГОТОВО:", n); break
        open(state, "w").write(after); time.sleep(3)

набор 10.17172/NOMAD/2024.05.10-1 → nomad_2024.05.10-1.csv
HTTP 502 <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.31.1</center>
</body>
</
  строк: 20
HTTP 429 {"detail":"Too many requests to a rate-limited endpoint. Only one in-flight connection is allowed, and new connections allowed every 5 seconds."}
  строк: 40
  строк: 60
  строк: 80
  строк: 100
HTTP 429 {"detail":"Too many requests to a rate-limited endpoint. Only one in-flight connection is allowed, and new connections allowed every 5 seconds."}
  строк: 120
  строк: 140
  строк: 160
  строк: 180
  строк: 200
  строк: 220
  строк: 240
  строк: 260
  строк: 280
  строк: 300
  строк: 320
  строк: 340
  строк: 360
  строк: 380
  строк: 400
  строк: 420
  строк: 440
  строк: 460
HTTP 429 {"detail":"Too many requests to a rate-limited endpoint. Only one in-flight connection is allowed, and new connections allowed every 5 seconds."}
  строк: 480
  строк: 500
H

In [4]:
import requests, csv, json, time, os
base = "https://nomad-lab.eu/prod/v1/api/v1"
OUT, STATE = "schumann_nomad_energies.csv", "nomad_cursor.txt"
body = {
    "owner": "public",
    "query": {"datasets.dataset_id": "180xLFvHQGuFNSfziBHiEA"},
    "pagination": {"page_size": 20},
    "required": {"metadata": "*", "run": {"calculation[-1]": {"energy": "*"}}},
}
if not os.path.exists(OUT):
    with open(OUT, "w", newline="") as f:
        csv.writer(f).writerow(("entry_id", "mainfile", "E_total_J", "E_total_eV"))
after = open(STATE).read().strip() if os.path.exists(STATE) else None
s = requests.Session()
total = sum(1 for _ in open(OUT)) - 1
while True:
    if after:
        body["pagination"]["page_after_value"] = after
    d = None
    for attempt in range(10):
        try:
            resp = s.post(base + "/entries/archive/query", json=body, timeout=600)
            if resp.status_code == 200:
                d = resp.json()
                break
            print("HTTP", resp.status_code, resp.text[:200])
        except Exception as ex:
            print("сбой:", type(ex).__name__)
        wait = 15 * (attempt + 1)
        print(f"жду {wait} с и повторяю")
        time.sleep(wait)
    if d is None:
        print("сервер не отвечает; запустите ячейку ещё раз позже — продолжит с места остановки")
        break
    new = []
    for x in d.get("data", []):
        a = x.get("archive", {})
        mf = a.get("metadata", {}).get("mainfile", "")
        try:
            e_J = a["run"][0]["calculation"][-1]["energy"]["total"]["value"]
            e_eV = e_J / 1.602176634e-19
        except Exception:
            e_J, e_eV = "", ""
        new.append((x.get("entry_id", ""), mf, e_J, e_eV))
    with open(OUT, "a", newline="") as f:
        csv.writer(f).writerows(new)
    total += len(new)
    after = d.get("pagination", {}).get("next_page_after_value")
    print("сохранено строк:", total)
    if not after:
        if os.path.exists(STATE):
            os.remove(STATE)
        print("ГОТОВО:", total, "строк")
        break
    open(STATE, "w").write(after)
    time.sleep(3)

сохранено строк: 20
сохранено строк: 40
HTTP 429 {"detail":"Too many requests to a rate-limited endpoint. Only one in-flight connection is allowed, and new connections allowed every 5 seconds."}
жду 15 с и повторяю
сохранено строк: 60
сохранено строк: 80
сохранено строк: 100
сохранено строк: 120
HTTP 429 {"detail":"Too many requests to a rate-limited endpoint. Only one in-flight connection is allowed, and new connections allowed every 5 seconds."}
жду 15 с и повторяю
сохранено строк: 140
сохранено строк: 160
сохранено строк: 180
HTTP 429 {"detail":"Too many requests to a rate-limited endpoint. Only one in-flight connection is allowed, and new connections allowed every 5 seconds."}
жду 15 с и повторяю
сохранено строк: 200
сохранено строк: 220
сохранено строк: 240
сохранено строк: 260
сохранено строк: 280
сохранено строк: 300
сохранено строк: 320
сохранено строк: 340
сохранено строк: 360
HTTP 429 {"detail":"Too many requests to a rate-limited endpoint. Only one in-flight connection is al

In [2]:
import requests, csv, json, time, re
url = "https://api.catalysis-hub.org/graphql"
HOSTS = {"Au", "Ag", "Cu"}
DOPANTS = ["Sn", "Bi", "In", "Ga", "Zn", "Ru", "Pd", "Pt", "Ni", "Co", "Fe", "Cu", "Ag", "Au"]
q = """query($comp: String, $after: String) {
  reactions(surfaceComposition: $comp, first: 200, after: $after) {
    totalCount pageInfo { hasNextPage endCursor }
    edges { node { pubId surfaceComposition facet Equation reactionEnergy sites dftFunctional } } } }"""
seen, rows = set(), []
for X in DOPANTS:
    after, got = None, 0
    while True:
        for attempt in range(8):
            try:
                r = requests.post(url, json={"query": q, "variables": {"comp": "~" + X, "after": after}}, timeout=300)
                d = r.json()
                if r.status_code == 200 and "errors" not in d: break
                print("HTTP", r.status_code, json.dumps(d)[:200])
            except Exception as ex:
                print("сбой:", type(ex).__name__)
            time.sleep(10 * (attempt + 1))
        else:
            print("пропускаю", X); break
        R = d["data"]["reactions"]
        for e in R["edges"]:
            n = e["node"]
            els = set(re.findall(r"[A-Z][a-z]?", n["surfaceComposition"] or ""))
            key = (n["pubId"], n["surfaceComposition"], n["facet"], n["Equation"], n["sites"])
            if X in els and els & (HOSTS - {X}) and key not in seen:
                seen.add(key)
                rows.append([n["pubId"], n["surfaceComposition"], n["facet"], n["Equation"],
                             n["reactionEnergy"], n["sites"], n["dftFunctional"]])
        got += len(R["edges"])
        print(X, got, "/", R["totalCount"], "| подходящих всего:", len(rows))
        if not R["pageInfo"]["hasNextPage"]: break
        after = R["pageInfo"]["endCursor"]; time.sleep(1)
with open("cathub_coinage_dopants.csv", "w", newline="") as f:
    csv.writer(f).writerows([["pub", "surface", "facet", "equation", "E_reaction_eV", "sites", "functional"]] + rows)
print("ГОТОВО:", len(rows), "строк")

сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
пропускаю Sn
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
пропускаю Bi
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
пропускаю In
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
пропускаю Ga
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSONDecodeError
пропускаю Zn
сбой: JSONDecodeError
сбой: JSONDecodeError
сбой: JSOND

In [3]:
import requests
rec = requests.get("https://zenodo.org/api/records/15648227", timeout=120).json()
for f in rec.get("files", []):
    name, size, link = f["key"], f["size"], f["links"]["self"]
    print(f"{name}: {size/1e6:.1f} МБ")
    if size < 200e6:
        with open(name, "wb") as out:
            out.write(requests.get(link, timeout=600).content)
        print("  скачан")
    else:
        print("  слишком большой — пропускаю")

main_results.zip: 825.5 МБ
  слишком большой — пропускаю


In [5]:
import requests, csv, time, os, json
URL = "https://api.catalysis-hub.org/graphql"
PUB = "MamunHighT2019"
OUT, STATE = "cathub_mamun2019.csv", "cathub_cursor.txt"
s = requests.Session()

def ask(query, variables=None):
    for attempt in range(8):
        try:
            r = s.get(URL, params={"query": query,
                                   "variables": json.dumps(variables or {})},
                      timeout=300)
            if r.status_code == 200 and r.text.strip().startswith("{"):
                d = r.json()
                if "errors" not in d:
                    return d
            print("HTTP", r.status_code, "|", r.url[:120], "|", r.text[:300].replace("\n", " "))
        except Exception as ex:
            print("сбой:", type(ex).__name__, str(ex)[:200])
        time.sleep(15 * (attempt + 1))
    return None

test = ask('{ reactions(pubId: "%s", first: 1) { totalCount } }' % PUB)
if test is None:
    raise SystemExit("Catalysis-Hub не отвечает — пришлите скриншот строк выше")
total = test["data"]["reactions"]["totalCount"]
print("реакций в наборе:", total)

Q = """query($after: String) { reactions(pubId: "%s", first: 100, after: $after) {
  pageInfo { hasNextPage endCursor }
  edges { node { surfaceComposition facet Equation reactionEnergy sites dftFunctional } } } }""" % PUB
if not os.path.exists(OUT):
    with open(OUT, "w", newline="") as f:
        csv.writer(f).writerow(["surface", "facet", "equation", "E_reaction_eV", "sites", "functional"])
after = open(STATE).read().strip() if os.path.exists(STATE) else None
n = sum(1 for _ in open(OUT)) - 1
while True:
    d = ask(Q, {"after": after})
    if d is None:
        print("оборвалось — перезапустите ячейку, продолжит с места"); break
    R = d["data"]["reactions"]
    rows = [[e["node"][k] for k in ("surfaceComposition", "facet", "Equation",
             "reactionEnergy", "sites", "dftFunctional")] for e in R["edges"]]
    with open(OUT, "a", newline="") as f:
        csv.writer(f).writerows(rows)
    n += len(rows)
    print(n, "/", total)
    if not R["pageInfo"]["hasNextPage"]:
        if os.path.exists(STATE): os.remove(STATE)
        print("ГОТОВО:", n); break
    after = R["pageInfo"]["endCursor"]
    open(STATE, "w").write(after)
    time.sleep(1)

HTTP 404 | https://api.catalysis-hub.org/graphql?query=%7B+reactions%28pubId%3A+%22MamunHighT2019%22%2C+first%3A+1%29+%7B+totalCoun | <!DOCTYPE HTML> <html lang='en-us'>   <head>     <title>Not Found</title>   </head>   <body>     HTTP Status: 404 (not found)   </body> </html>
HTTP 404 | https://api.catalysis-hub.org/graphql?query=%7B+reactions%28pubId%3A+%22MamunHighT2019%22%2C+first%3A+1%29+%7B+totalCoun | <!DOCTYPE HTML> <html lang='en-us'>   <head>     <title>Not Found</title>   </head>   <body>     HTTP Status: 404 (not found)   </body> </html>
HTTP 404 | https://api.catalysis-hub.org/graphql?query=%7B+reactions%28pubId%3A+%22MamunHighT2019%22%2C+first%3A+1%29+%7B+totalCoun | <!DOCTYPE HTML> <html lang='en-us'>   <head>     <title>Not Found</title>   </head>   <body>     HTTP Status: 404 (not found)   </body> </html>
HTTP 404 | https://api.catalysis-hub.org/graphql?query=%7B+reactions%28pubId%3A+%22MamunHighT2019%22%2C+first%3A+1%29+%7B+totalCoun | <!DOCTYPE HTML> <html lang='en-u

KeyboardInterrupt: 

In [6]:
import requests, csv, time, os, json, tarfile
PUB = "MamunHighT2019"
OUT, STATE = "cathub_mamun2019.csv", "cathub_cursor.txt"
s = requests.Session()
ENDPOINTS = ["https://www.catalysis-hub.org/api/graphql",
             "https://api.catalysis-hub.org/api/graphql",
             "https://catalysis-hub.org/api/graphql",
             "https://api.catalysis-hub.org/graphql/"]

def try_once(url, method, query, variables):
    if method == "POST":
        return s.post(url, json={"query": query, "variables": variables}, timeout=120)
    return s.get(url, params={"query": query, "variables": json.dumps(variables)}, timeout=120)

TEST = '{ reactions(pubId: "%s", first: 1) { totalCount } }' % PUB
URL = METHOD = None
for u in ENDPOINTS:
    for m in ("POST", "GET"):
        try:
            r = try_once(u, m, TEST, {})
            ok = r.status_code == 200 and r.text.strip().startswith("{") and "errors" not in r.json()
            print(m, u, "->", r.status_code, "OK" if ok else r.text[:120].replace("\n", " "))
            if ok:
                URL, METHOD, total = u, m, r.json()["data"]["reactions"]["totalCount"]
                break
        except Exception as ex:
            print(m, u, "-> сбой", type(ex).__name__)
    if URL:
        break

if URL:
    print("работает:", METHOD, URL, "| реакций:", total)
    Q = """query($after: String) { reactions(pubId: "%s", first: 100, after: $after) {
      pageInfo { hasNextPage endCursor }
      edges { node { surfaceComposition facet Equation reactionEnergy sites dftFunctional } } } }""" % PUB
    if not os.path.exists(OUT):
        with open(OUT, "w", newline="") as f:
            csv.writer(f).writerow(["surface", "facet", "equation", "E_reaction_eV", "sites", "functional"])
    after = open(STATE).read().strip() if os.path.exists(STATE) else None
    n = sum(1 for _ in open(OUT)) - 1
    while True:
        d = None
        for attempt in range(6):
            try:
                r = try_once(URL, METHOD, Q, {"after": after})
                if r.status_code == 200 and r.text.strip().startswith("{"):
                    d = r.json()
                    if "errors" not in d:
                        break
                print("HTTP", r.status_code, r.text[:150])
            except Exception as ex:
                print("сбой:", type(ex).__name__)
            d = None
            time.sleep(15 * (attempt + 1))
        if d is None:
            print("оборвалось — перезапустите ячейку, продолжит с места"); break
        R = d["data"]["reactions"]
        rows = [[e["node"][k] for k in ("surfaceComposition", "facet", "Equation",
                 "reactionEnergy", "sites", "dftFunctional")] for e in R["edges"]]
        with open(OUT, "a", newline="") as f:
            csv.writer(f).writerows(rows)
        n += len(rows); print(n, "/", total)
        if not R["pageInfo"]["hasNextPage"]:
            if os.path.exists(STATE): os.remove(STATE)
            print("ГОТОВО:", n); break
        after = R["pageInfo"]["endCursor"]; open(STATE, "w").write(after); time.sleep(1)
else:
    print("\nCatalysis-Hub недоступен — смотрю архив на Materials Cloud\n")
    MC = "https://archive.materialscloud.org/records/xxdhq-f1v83/files/"
    for fn in ("README.txt", "files_description.md"):
        print("=====", fn); print(requests.get(MC + fn + "?download=1", timeout=120).text)
    print("===== первые файлы H.tar.gz (без полной загрузки)")
    with requests.get(MC + "H.tar.gz?download=1", stream=True, timeout=600) as r:
        r.raw.decode_content = True
        with tarfile.open(fileobj=r.raw, mode="r|gz") as t:
            for i, m in enumerate(t):
                print(f"{m.size/1e3:10.1f} kB  {m.name}")
                if i >= 60:
                    break

POST https://www.catalysis-hub.org/api/graphql -> 404 Cannot POST /api/graphql 
GET https://www.catalysis-hub.org/api/graphql -> 200 <!DOCTYPE html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-sca
POST https://api.catalysis-hub.org/api/graphql -> 405 Method Not Allowed 
GET https://api.catalysis-hub.org/api/graphql -> 404 Not Found 
POST https://catalysis-hub.org/api/graphql -> 405 Method Not Allowed 
GET https://catalysis-hub.org/api/graphql -> 404 Not Found 
POST https://api.catalysis-hub.org/graphql/ -> 405 Method Not Allowed 
GET https://api.catalysis-hub.org/graphql/ -> 404 

Catalysis-Hub недоступен — смотрю архив на Materials Cloud

===== README.txt
The dataset consists of a large number of Quantum Espresso log files, packed into separate tar.gz files. The files are divided into 6 categories, based on the chemical species adsorbed to the surface.

H.tar.gz: adsorption of H
C.tar.gz: adsorption of C, CH, CH2 and CH3
N.tar.g

In [7]:
!pip -q install ase
import requests, tarfile, io, os, collections
from ase.io import read, write

MC = "https://archive.materialscloud.org/records/xxdhq-f1v83/files/"
HOSTS = {"Au", "Ag", "Cu"}
GAS = {"H", "O", "N", "C", "S"}

def process(archive):
    out = archive.replace(".tar.gz", "_coinage.xyz")
    if os.path.exists(out): os.remove(out)
    seen, total, kept, bad = collections.Counter(), 0, 0, 0
    with requests.get(MC + archive + "?download=1", stream=True, timeout=1800) as r:
        r.raw.decode_content = True
        with tarfile.open(fileobj=r.raw, mode="r|gz") as t:
            for m in t:
                base = os.path.basename(m.name)
                if not m.isfile() or base.startswith("._") or not base.endswith((".log", ".out", ".pwo")):
                    continue
                total += 1
                txt = t.extractfile(m).read().decode("utf-8", "ignore")
                try:
                    a = read(io.StringIO(txt), format="espresso-out", index=-1)
                    e = a.get_potential_energy()
                except Exception:
                    bad += 1
                    continue
                metals = set(a.get_chemical_symbols()) - GAS
                seen.update(metals)
                if (metals & HOSTS) or not metals:
                    a.info["source"] = m.name
                    a.info["energy_eV"] = e
                    write(out, a, format="extxyz", append=True)
                    kept += 1
                if total % 500 == 0:
                    print(f"  {archive}: прочитано {total}, сохранено {kept}, не прочиталось {bad}")
    print(f"{archive} ГОТОВО: всего {total}, сохранено {kept}, не прочиталось {bad}")
    print("  металлы в архиве:", " ".join(sorted(seen)))

for arch in ["references.tar.gz", "H.tar.gz", "O.tar.gz", "N.tar.gz"]:
    process(arch)

!zip -q mamun_coinage.zip *_coinage.xyz && ls -lh mamun_coinage.zip

  references.tar.gz: прочитано 500, сохранено 63, не прочиталось 84
  references.tar.gz: прочитано 1000, сохранено 110, не прочиталось 200
  references.tar.gz: прочитано 1500, сохранено 218, не прочиталось 271
  references.tar.gz: прочитано 2500, сохранено 324, не прочиталось 396
  references.tar.gz: прочитано 3000, сохранено 399, не прочиталось 401
  references.tar.gz: прочитано 4000, сохранено 589, не прочиталось 413
references.tar.gz ГОТОВО: всего 4062, сохранено 599, не прочиталось 414
  металлы в архиве: Ag Al Au Bi Cd Co Cr Cu Fe Ga Hf Hg In Ir La Mn Mo Nb Ni Os Pb Pd Pt Re Rh Ru Sc Sn Ta Tc Ti Tl V W Y Zn Zr
  H.tar.gz: прочитано 500, сохранено 164, не прочиталось 0
  H.tar.gz: прочитано 1000, сохранено 275, не прочиталось 79
  H.tar.gz: прочитано 1500, сохранено 335, не прочиталось 184
  H.tar.gz: прочитано 2000, сохранено 414, не прочиталось 281
  H.tar.gz: прочитано 2500, сохранено 476, не прочиталось 350
  H.tar.gz: прочитано 3000, сохранено 548, не прочиталось 423


KeyboardInterrupt: 

In [8]:
import requests, tarfile, io, os, csv, collections
from concurrent.futures import ProcessPoolExecutor
from ase.io import read

MC = "https://archive.materialscloud.org/records/xxdhq-f1v83/files/"
GAS = {"H", "O", "N", "C", "S"}
COLS = ["archive", "source", "formula", "metals", "natoms", "a", "b", "c", "gamma",
        "energy_eV", "adsorbate", "binding_atom_neighbors", "top_layer"]

def summarize(a):
    sym = a.get_chemical_symbols(); pos = a.get_positions()
    met = [i for i, s in enumerate(sym) if s not in GAS]
    ads = [i for i, s in enumerate(sym) if s in GAS]
    near = top = ""
    if met:
        zmax = max(pos[i][2] for i in met)
        top = " ".join(f"{k}{v}" for k, v in sorted(collections.Counter(
            sym[i] for i in met if pos[i][2] > zmax - 1.0).items()))
    if ads and met:
        b = min(ads, key=lambda i: pos[i][2])
        d = a.get_distances(b, met, mic=True)
        dmin = d.min()
        near = ",".join(sorted(sym[met[k]] for k in range(len(met)) if d[k] < 1.15 * dmin))
    cp = a.cell.cellpar()
    metals = " ".join(f"{k}{v}" for k, v in sorted(collections.Counter(sym[i] for i in met).items()))
    adsorb = "".join(sorted(sym[i] for i in ads))
    return [a.get_chemical_formula(mode="hill"), metals, len(a),
            round(cp[0], 3), round(cp[1], 3), round(cp[2], 3), round(cp[5], 1), adsorb, near, top]

def process(archive):
    out = archive.replace(".tar.gz", "_all.csv")
    total = kept = bad = 0
    with open(out, "w", newline="") as fo:
        w = csv.writer(fo); w.writerow(COLS)
        with requests.get(MC + archive + "?download=1", stream=True, timeout=1800) as r:
            r.raw.decode_content = True
            with tarfile.open(fileobj=r.raw, mode="r|gz") as t:
                for m in t:
                    base = os.path.basename(m.name)
                    if not m.isfile() or base.startswith("._") or not base.endswith((".log", ".out", ".pwo")):
                        continue
                    total += 1
                    txt = t.extractfile(m).read().decode("utf-8", "ignore")
                    try:
                        a = read(io.StringIO(txt), format="espresso-out", index=-1)
                        e = a.get_potential_energy()
                        w.writerow([archive, m.name] + summarize(a)[:7] + [round(e, 5)] + summarize(a)[7:])
                        kept += 1
                    except Exception:
                        bad += 1
                    if total % 1000 == 0:
                        print(f"  {archive}: просмотрено {total}, сохранено {kept}", flush=True)
    return f"{archive} ГОТОВО: всего {total}, сохранено {kept}, не прочиталось {bad}"

with ProcessPoolExecutor(4) as ex:
    for msg in ex.map(process, ["references.tar.gz", "H.tar.gz", "O.tar.gz", "N.tar.gz"]):
        print(msg, flush=True)

  references.tar.gz: просмотрено 1000, сохранено 800
  references.tar.gz: просмотрено 2000, сохранено 1615
  N.tar.gz: просмотрено 1000, сохранено 817
  O.tar.gz: просмотрено 1000, сохранено 806
  references.tar.gz: просмотрено 3000, сохранено 2599
  references.tar.gz: просмотрено 4000, сохранено 3587
references.tar.gz ГОТОВО: всего 4062, сохранено 3648, не прочиталось 414
  H.tar.gz: просмотрено 1000, сохранено 921
  O.tar.gz: просмотрено 2000, сохранено 1636
  N.tar.gz: просмотрено 2000, сохранено 1649
  H.tar.gz: просмотрено 2000, сохранено 1719
  O.tar.gz: просмотрено 3000, сохранено 2481
  N.tar.gz: просмотрено 3000, сохранено 2318
  H.tar.gz: просмотрено 3000, сохранено 2577
  N.tar.gz: просмотрено 4000, сохранено 3045
  O.tar.gz: просмотрено 4000, сохранено 3343
  H.tar.gz: просмотрено 4000, сохранено 3411
  N.tar.gz: просмотрено 5000, сохранено 3737
  O.tar.gz: просмотрено 5000, сохранено 4148
  H.tar.gz: просмотрено 5000, сохранено 4313
  N.tar.gz: просмотрено 6000, сохранено 

In [9]:
!zip -q mamun_all.zip references_all.csv H_all.csv O_all.csv N_all.csv && ls -lh mamun_all.zip

-rw-r--r-- 1 root root 901K Oct 29 23:20 mamun_all.zip
